# 📑 O-ISAC ComstPrev Digitzation (Phase 1 Only)

**Hedef:** `data/cprev` klasöründeki PDF dosyalarını metne (Markdown) çevirmek.
**Kapsam:** Sadece Faz 1 (Görsel analiz veya LLM/Gemini yoktur).

---
**Klasörler:**
- **Girdi (PDF):** `data/cprev` (77 Dosya)
- **Çıktı (MD):** `data/proc_markdowns_comstPrev`
- **Sonuç:** `data/ext_res_comstPrev_v3`

In [ ]:
# @title 1. Kurulum ve Ayarlar

# 1. Gerekli kütüphaneleri yükle
!pip uninstall -y marker marker-pdf numpy -q
!pip install "numpy<2.0" -q
!pip install marker-pdf --upgrade --force-reinstall -q
!pip install transformers torch pillow -q

# 2. Google Drive Bağla
from google.colab import drive
import os
import sys

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 3. Proje Yollarını Tanımla (ŞEFFAF AYARLAR)
PROJECT_ROOT = '/content/drive/MyDrive/AKU_WorkSpace/survey_fdgit/OISAC_PRISMA_COMST'
PDF_DIR = os.path.join(PROJECT_ROOT, 'data/cprev')
MARKDOWN_DIR = os.path.join(PROJECT_ROOT, 'data/proc_markdowns_comstPrev')
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'data/ext_res_comstPrev_v3')

# Python yoluna ekle
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'analysis/nb'))
sys.path.insert(0, PROJECT_ROOT)

print(f"📂 Hedef PDF Klasörü: {PDF_DIR}")
print(f"📝 Hedef Markdown Klasörü: {MARKDOWN_DIR}")
print("✅ Kurulum Tamam.")

In [ ]:
# @title 2. Pipeline Başlatma (Ayarları Uygula)
import extraction_pipeline_v3 as v3
import importlib
import glob

# Kodu tazelemek için reload yapıyoruz
importlib.reload(v3)

# --- KRİTİK ADIM: Ayarları 'v3' modülü üzerinde doğrudan değiştiriyoruz ---
# Bu sayede arka plandaki kod, bizim belirlediğimiz bu yeni klasörleri kullanıyor.
v3.Config.PDF_DIR = PDF_DIR
v3.Config.MARKDOWN_DIR = MARKDOWN_DIR
v3.Config.OUTPUT_DIR = OUTPUT_DIR
v3.Config.CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, "checkpoint.json")

# Klasörleri oluştur
v3.Config.init_dirs()
checkpoint = v3.CheckpointManager(v3.Config.CHECKPOINT_FILE)

# Kontrol Edelim
pdf_files = glob.glob(os.path.join(PDF_DIR, '*.pdf'))
print(f"🎯 Hedeflenen PDF Sayısı: {len(pdf_files)}")
print(f"   (Beklenen sayı 77 olmalı. Eğer 221 ise yanlış klasöre bakıyoruz demektir.)")

In [ ]:
# @title 3. Çalıştır: PDF -> Markdown
print("🚀 İşlem Başlıyor... Sadece PDF dönüşümü yapılacak.")

# Force_all=False dersek, sadece yeni dosyaları işler. 
# True dersek hepsini baştan yapar.
v3.phase1_marker_conversion(checkpoint, force_all=False)

print("✅ İşlem Tamamlandı.")